In [18]:
# Instala o pacote VGAM se ele ainda não estiver instalado
library(tidyverse)    # Data wrangling and visualization
library(VGAM)
library(survey)       # Survey analysis
library(readr)        # CSV reading
library(svyVGAM)
library(dplyr)
library(broom)
library(scales)
library(tidyr)
library(rlang)
library(purrr)
library(kableExtra)
library(emmeans)
library(readr)
library(srvyr)
library(knitr)

library(tibble)

library(car)
library(corrplot)

In [19]:
### 02/11/2025
# Adicionar as variaveis nas analises
# Peso ao nascer e prematuro
# Cesaria

In [20]:
# Carregando os dados
df_maes_grupo_2013 <- read_csv("data/df_maes_grupo_2013.csv")
df_maes_grupo_2019 <- read_csv("data/df_maes_grupo_2019.csv")

Rows: 8072 Columns: 40
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (16): regiao_metropolitana, capital_metropolitana, sexo, estado_civil, U...
dbl (24): V0024, UPA_PNS, V00291, UF, dia_nascimento, mes_nascimento, ano_na...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10575 Columns: 40
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (16): regiao_metropolitana, capital_metropolitana, sexo, estado_civil, U...
dbl (24): V0024, UPA_PNS, V00291, UF, dia_nascimento, mes_nascimento, ano_na...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


# Peso amostral e Desenho da Pesquisa

In [21]:
library(dplyr)
library(survey)

# Função: prepara um dataframe, aplica fatores e cria o svydesign
prepare_year_design <- function(df,
                                year,
                                weight_expr = NULL,
                                weight_name = "peso_morador_selec") { # Removido keep_ids pois não era usado
  # cópia segura
  df0 <- df
  
  # --- NOVO BLOCO: DECODIFICAÇÃO E APLICAÇÃO DE FATORES ---
  # Este passo é executado antes de qualquer outra coisa para garantir
  # que as variáveis estejam prontas para a análise.
  message("Aplicando decodificação e ordenação de fatores...")
  df0 <- df0 %>%
    mutate(
      # Escolaridade (Referência = 'Fundamental')
      escolaridade_fct = factor(escolaridade, levels = c("Fundamental", "Médio", "Superior", "Pós-graduação")),
      
      # Raça/Cor (Referência = 'Branca')
      raca_fct = factor(raca, levels = c("Branca", "Parda", "Preta", "Amarela", "Indígena")),

      regiao_metropolitana_fct = factor(regiao_metropolitana, levels = c("Urbano", "Rural")),
      
      # Região (Referência = 'Sudeste')
      regiao_brasileira_fct = factor(regiao_brasileira, levels = c("Sudeste", "Norte", "Nordeste", "Sul", "Centro-Oeste")),
      
      # Estado Civil (Decodifica e define Referência = 'Solteiro(a)')
      estado_civil_fct = factor(estado_civil, levels = c("Solteiro", "Casado", "Divorciado", "Viúvo")),

       # Renda (Decodifica e define Referência = '<1 SM')
      faixa_renda_per_capta_fct = factor(faixa_renda_per_capta, levels = c("<1 SM", "1 a <2 SM", "2 a <3 SM", "3 a <5 SM", "≥5 SM")),

      # IMC Classificação (Referência = 'Normal')
      imc_classificacao_fct = factor(imc_classificacao, levels = c("Sobrepeso", "Obesidade", "Magreza", "Normal", "Obesidade Grave")),

      # Comorbidades (Referência = 'Não')
      comorbidade_fct = factor(comorbidade, levels = c("Não", "Sim")),

      # Atividade Física (Referência = 'Sim')
      atividade_fisica_fct = factor(atividade_fisica, levels = c("Sim", "Não")),

      # pre_natal_adequado (Referência = 'Sim')
      pre_natal_adequado_fct = factor(pre_natal_adequado, levels = c("Sim", "Não")),

      # peso_nascer_inferior_2_5 (Referência = 'Sim')
      peso_nascer_inferior_2_5_fct = factor(peso_nascer_inferior_2_5, levels = c("Sim", "Não")),

      # prematuro (Referência = 'Sim')
      prematuro_fct = factor(prematuro, levels = c("Sim", "Não")),

      # cesaria (Referência = 'Sim')
      cesaria_fct = factor(cesaria, levels = c("Sim", "Não")),
    )


  # -----------------------------------------------------------
  
  # 1) calcula o peso
  if (!is.null(weight_expr)) {
    df0 <- df0 %>%
      mutate(V00291 = suppressWarnings(as.numeric(as.character(V00291))))
    df0[[weight_name]] <- eval(parse(text = weight_expr), envir = df0)
  } else {
    if (weight_name %in% names(df0)) df0[[weight_name]] <- suppressWarnings(as.numeric(df0[[weight_name]]))
  }
  
  # 2) coerção de UPA_PNS e V0024 (estrato)
  df0 <- df0 %>%
    mutate(
      UPA_PNS_num = suppressWarnings(as.integer(trimws(as.character(UPA_PNS)))),
      STRATO_num  = suppressWarnings(as.integer(trimws(as.character(V0024))))
    )
  
  # 3) filtrar observações com informações essenciais
  df_clean <- df0 %>%
    filter(
      !is.na(UPA_PNS_num),
      !is.na(STRATO_num),
      !is.na(.data[[weight_name]]),
      !is.na(parto_idade_avancada)
    ) %>%
    mutate(
      ano = as.integer(year),
      UPA_PNS_id = paste0(year, "_", UPA_PNS_num),
      estrato_id = paste0(year, "_", STRATO_num)
    )
  
  # 4) criar objeto svydesign para este ano
  options(survey.lonely.psu = "adjust")
  design_obj <- svydesign(
    ids = ~UPA_PNS_num,
    strata = ~STRATO_num,
    weights = as.formula(paste0("~", weight_name)),
    data = df_clean,
    nest = TRUE
  )
  
  # 5) mensagens de verificação
  message("Ano: ", year)
  message("  Observações originais: ", nrow(df))
  message("  Observações após limpeza: ", nrow(df_clean))
  message("  Somatório de pesos (população projetada) : ", format(sum(df_clean[[weight_name]], na.rm = TRUE), big.mark = ","))
  message("  Somatório de parto_idade_avancada : ", format(sum(df_clean$parto_idade_avancada, na.rm = TRUE), big.mark = ","))
  
  # retorno como lista (dataframe limpo + design)
  list(df = df_clean, design = design_obj)
}

# ============================
# Exemplo de uso com seus multiplicadores já estabelecidos:
# (faça isso para 2013 e 2019 separadamente)
# ============================

# Para 2013 (substitua pelo nome original do df 2013)
res2013 <- prepare_year_design(
  df = df_maes_grupo_2013,
  year = 2013,
  weight_expr = "V00291 * (60202 / 145572211)",
  weight_name = "peso_morador_selec"
)

design_maes_2013 <- res2013$design
df_maes_grupo_2013_clean <- res2013$df

# Para 2019 (substitua pelo nome original do df 2019)
res2019 <- prepare_year_design(
  df = df_maes_grupo_2019,
  year = 2019,
  weight_expr = "V00291 * (94114 / 168426190)",
  weight_name = "peso_morador_selec"
)

design_maes_2019 <- res2019$design
df_maes_grupo_2019_clean <- res2019$df

Aplicando decodificação e ordenação de fatores...

Ano: 2013

  Observações originais: 8072

  Observações após limpeza: 8072

  Somatório de pesos (população projetada) : 6,615.056

  Somatório de parto_idade_avancada : 1,078

Aplicando decodificação e ordenação de fatores...

Ano: 2019

  Observações originais: 10575

  Observações após limpeza: 10575

  Somatório de pesos (população projetada) : 10,743.6

  Somatório de parto_idade_avancada : 2,130



## Efeito entre subgrupos

In [22]:
library(survey)
library(dplyr)
library(tibble)
library(knitr)

gerar_tabela_or_evolucao <- function(df_ano1, df_ano2,
                                    ano1_label = "2013", ano2_label = "2019",
                                    weight_name = "peso_morador_selec",
                                    variavel_desfecho,   # 0 = Young, 1 = AMA (ou "1")
                                    variavel_preditora,  # ex: regiao_metropolitana_fct
                                    ordem_referencia,    # ordem dos níveis do agrupador
                                    id = "UPA_PNS_num",
                                    strata = "STRATO_num") {

  # --- preparar dados combinados ---
  df_combinado <- bind_rows(
    df_ano1 %>% mutate(ano = ano1_label),
    df_ano2 %>% mutate(ano = ano2_label)
  ) %>%
    mutate(
      !!variavel_preditora := factor(.data[[variavel_preditora]], levels = ordem_referencia),
      ano_fct = factor(ano, levels = c(ano1_label, ano2_label))
    )

  # criar ..y_bin com 1 = AMA
  y_raw <- df_combinado[[variavel_desfecho]]
  if (is.numeric(y_raw)) {
    df_combinado$..y_bin <- as.numeric(y_raw == 1)
  } else {
    df_combinado$..y_bin <- as.numeric(as.character(y_raw) == "1")
    if (all(is.na(df_combinado$..y_bin))) {
      stop("variavel_desfecho não numérica e não contém '1' como nível. Passe 1 = AMA.")
    }
  }

  # svydesign combinado
  design_combinado <- svydesign(
    ids = as.formula(paste0("~", id)),
    strata = as.formula(paste0("~", strata)),
    weights = as.formula(paste0("~", weight_name)),
    data = df_combinado,
    nest = TRUE
  )

  # --- Função auxiliar corrigida: calcula pAMA = P(grupo | AMA) e pYoung = P(grupo | Young)
  calc_for_level_and_year <- function(des, level, ano_label) {
    # IMPORTANT: subset apenas por ano (não por grupo)
    des_year <- subset(des, ano_fct == ano_label)
    if (nrow(des_year$variables) == 0) return(NULL)

    # construir indicadores no contexto do ANO (não do grupo):
    des_year$variables$..joint_A <- as.numeric(des_year$variables[[variavel_preditora]] == level & des_year$variables$..y_bin == 1)
    des_year$variables$..joint_Y <- as.numeric(des_year$variables[[variavel_preditora]] == level & des_year$variables$..y_bin == 0)
    des_year$variables$..total_A <- as.numeric(des_year$variables$..y_bin == 1)
    des_year$variables$..total_Y <- as.numeric(des_year$variables$..y_bin == 0)

    # estimar as quatro quantidades (proporções na população do ano)
    obj <- tryCatch(
      svymean(~ I(..joint_A) + I(..joint_Y) + I(..total_A) + I(..total_Y),
             design = des_year, na.rm = TRUE),
      error = function(e) NULL
    )
    if (is.null(obj)) return(NULL)

    est <- coef(obj)
    V <- vcov(obj)
    nm <- names(est)
    idx_jA <- grep("joint_A", nm)[1]
    idx_jY <- grep("joint_Y", nm)[1]
    idx_tA <- grep("total_A", nm)[1]
    idx_tY <- grep("total_Y", nm)[1]

    jA <- as.numeric(est[idx_jA]); jY <- as.numeric(est[idx_jY])
    tA <- as.numeric(est[idx_tA]); tY <- as.numeric(est[idx_tY])

    # se denominadores inválidos, retornar NA
    if (is.na(tA) || is.na(tY) || tA <= 0 || tY <= 0) {
      return(list(pAMA = NA, pYoung = NA, var_pAMA = NA, var_pYoung = NA, cov_pApY = NA, df = degf(des_year)))
    }

    pA <- jA / tA
    pY <- jY / tY

    # delta method para var(pA), var(pY) e cov(pA,pY)
    # theta = (jA, jY, tA, tY)
    J_pA <- c(1 / tA, 0, -jA / (tA^2), 0)
    J_pY <- c(0, 1 / tY, 0, -jY / (tY^2))

    Vsub <- V[c(idx_jA, idx_jY, idx_tA, idx_tY), c(idx_jA, idx_jY, idx_tA, idx_tY), drop = FALSE]

    var_pA <- as.numeric( J_pA %*% Vsub %*% J_pA )
    var_pY <- as.numeric( J_pY %*% Vsub %*% J_pY )
    cov_pApY <- as.numeric( J_pA %*% Vsub %*% J_pY )

    return(list(pAMA = pA, pYoung = pY,
                var_pAMA = var_pA, var_pYoung = var_pY, cov_pApY = cov_pApY,
                df = degf(des_year)))
  } # fim calc_for_level_and_year

  # --- delta on logit to get OR ---
  calc_OR_from_p <- function(pA, pY, var_pA, var_pY, cov_pApY, df_sub) {
    if (is.na(pA) || is.na(pY) || pA <= 0 || pA >= 1 || pY <= 0 || pY >= 1) {
      return(list(OR = NA, ci = c(NA, NA), p = NA, logOR = NA, se_logOR = NA))
    }
    gA <- 1 / (pA * (1 - pA)); gY <- 1 / (pY * (1 - pY))
    var_logitA <- (gA^2) * var_pA
    var_logitY <- (gY^2) * var_pY
    cov_logitA_logitY <- gA * gY * cov_pApY
    var_logOR <- var_logitA + var_logitY - 2 * cov_logitA_logitY
    if (var_logOR < 0) var_logOR <- NA_real_
    se_logOR <- if (!is.na(var_logOR)) sqrt(var_logOR) else NA_real_
    logOR <- log(pA / (1 - pA)) - log(pY / (1 - pY))
    if (!is.na(df_sub) && df_sub > 0) {
      crit <- qt(0.975, df_sub)
      pval <- 2 * pt(-abs(logOR / se_logOR), df_sub)
    } else {
      crit <- qnorm(0.975)
      pval <- 2 * (1 - pnorm(abs(logOR / se_logOR)))
    }
    ci_log <- logOR + c(-1, 1) * crit * se_logOR
    OR <- exp(logOR)
    ci_OR <- exp(ci_log)
    list(OR = OR, ci = ci_OR, p = pval, logOR = logOR, se_logOR = se_logOR)
  }

  # --- loop por níveis ---
  resultados <- list()
  for (nivel in ordem_referencia) {
    e1 <- calc_for_level_and_year(design_combinado, nivel, ano1_label)
    e2 <- calc_for_level_and_year(design_combinado, nivel, ano2_label)

    r1 <- if (is.null(e1)) list(pYoung = NA, pAMA = NA, OR = NA, ci = c(NA, NA), p = NA, logOR = NA, se = NA, df = NA) else {
      tmp <- calc_OR_from_p(e1$pAMA, e1$pYoung, e1$var_pAMA, e1$var_pYoung, e1$cov_pApY, e1$df)
      list(pYoung = e1$pYoung, pAMA = e1$pAMA, OR = tmp$OR, ci = tmp$ci, p = tmp$p, logOR = tmp$logOR, se = tmp$se_logOR, df = e1$df,
           var_pY = e1$var_pYoung, var_pA = e1$var_pAMA)
    }

    r2 <- if (is.null(e2)) list(pYoung = NA, pAMA = NA, OR = NA, ci = c(NA, NA), p = NA, logOR = NA, se = NA, df = NA) else {
      tmp <- calc_OR_from_p(e2$pAMA, e2$pYoung, e2$var_pAMA, e2$var_pYoung, e2$cov_pApY, e2$df)
      list(pYoung = e2$pYoung, pAMA = e2$pAMA, OR = tmp$OR, ci = tmp$ci, p = tmp$p, logOR = tmp$logOR, se = tmp$se_logOR, df = e2$df,
           var_pY = e2$var_pYoung, var_pA = e2$var_pAMA)
    }

    # p_change: diferença dos logORs assumindo independência entre anos (padrão para inquéritos distintos)
    if (!is.na(r1$logOR) && !is.na(r2$logOR) && !is.na(r1$se) && !is.na(r2$se)) {
      var_change <- (r1$se^2) + (r2$se^2)
      se_change <- sqrt(var_change)
      if (!is.na(r1$df) && !is.na(r2$df) && r1$df > 0 && r2$df > 0) {
        df_use <- min(r1$df, r2$df)
        p_change <- 2 * pt(-abs((r2$logOR - r1$logOR) / se_change), df_use)
      } else {
        p_change <- 2 * (1 - pnorm(abs((r2$logOR - r1$logOR) / se_change)))
      }
    } else {
      p_change <- NA
    }

    # formatação
    fmt_pct <- function(p) ifelse(is.na(p), NA_character_, sprintf("%.1f", 100 * p))
    fmt_ci_pct <- function(ci) ifelse(any(is.na(ci)), NA_character_, sprintf("[%.1f - %.1f]", 100 * ci[1], 100 * ci[2]))
    fmt_orci <- function(or, ci) ifelse(is.na(or), NA_character_, sprintf("%.2f (%.2f–%.2f)", or, ci[1], ci[2]))
    fmt_p <- function(p) {
      if (is.na(p)) return(NA_character_)
      if (p < 0.001) return("<0.001*")
      pstr <- format(round(p, 3), nsmall = 3)
      if (p <= 0.05) return(paste0(pstr, "*"))
      return(pstr)
    }

    # construir CIs para p usando var armazenada (se disponível)
    ci_pY_2013 <- NA; ci_pA_2013 <- NA; ci_pY_2019 <- NA; ci_pA_2019 <- NA
    if (!is.null(e1) && !is.na(e1$var_pY)) {
      se_pY <- sqrt(e1$var_pY); z <- if (!is.na(e1$df) && e1$df>0) qt(0.975, e1$df) else qnorm(0.975)
      ci_pY_2013 <- pmax(0, pmin(1, c(r1$pYoung - z*se_pY, r1$pYoung + z*se_pY)))
    }
    if (!is.null(e1) && !is.na(e1$var_pA)) {
      se_pA <- sqrt(e1$var_pA); z <- if (!is.na(e1$df) && e1$df>0) qt(0.975, e1$df) else qnorm(0.975)
      ci_pA_2013 <- pmax(0, pmin(1, c(r1$pAMA - z*se_pA, r1$pAMA + z*se_pA)))
    }
    if (!is.null(e2) && !is.na(e2$var_pY)) {
      se_pY <- sqrt(e2$var_pY); z <- if (!is.na(e2$df) && e2$df>0) qt(0.975, e2$df) else qnorm(0.975)
      ci_pY_2019 <- pmax(0, pmin(1, c(r2$pYoung - z*se_pY, r2$pYoung + z*se_pY)))
    }
    if (!is.null(e2) && !is.na(e2$var_pA)) {
      se_pA <- sqrt(e2$var_pA); z <- if (!is.na(e2$df) && e2$df>0) qt(0.975, e2$df) else qnorm(0.975)
      ci_pA_2019 <- pmax(0, pmin(1, c(r2$pAMA - z*se_pA, r2$pAMA + z*se_pA)))
    }

    resultados[[nivel]] <- tibble::tibble(
      Categoria = nivel,
      `% Young (2013)` = fmt_pct(r1$pYoung),
      `IC 95% (Young) 2013` = ifelse(is.null(ci_pY_2013), NA_character_, fmt_ci_pct(ci_pY_2013)),
      `% AMA (2013)` = fmt_pct(r1$pAMA),
      `IC 95% (AMA) 2013` = ifelse(is.null(ci_pA_2013), NA_character_, fmt_ci_pct(ci_pA_2013)),
      `OR (95% CI) 2013` = fmt_orci(r1$OR, r1$ci),
      `p-valor 2013` = fmt_p(r1$p),

      `% Young (2019)` = fmt_pct(r2$pYoung),
      `IC 95% (Young) 2019` = ifelse(is.null(ci_pY_2019), NA_character_, fmt_ci_pct(ci_pY_2019)),
      `% AMA (2019)` = fmt_pct(r2$pAMA),
      `IC 95% (AMA) 2019` = ifelse(is.null(ci_pA_2019), NA_character_, fmt_ci_pct(ci_pA_2019)),
      `OR (95% CI) 2019` = fmt_orci(r2$OR, r2$ci),
      `p-valor 2019` = fmt_p(r2$p),

      `p_change (2013->2019)` = fmt_p(p_change)
    )
  } # fim loop

  tabela_final <- bind_rows(resultados)

  cat("\n==================================================================\n")
  cat(" Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)\n")
  cat(" Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta\n")
  cat("==================================================================\n\n")

  print(kable(tabela_final, align = "l"))


  # --- Teste de Rao–Scott por ano (adição nova) ---
  design_ano1 <- subset(design_combinado, ano_fct == ano1_label)
  design_ano2 <- subset(design_combinado, ano_fct == ano2_label)

  teste_rao_ano1 <- tryCatch(
    svychisq(as.formula(paste0("~", variavel_desfecho, " + ", variavel_preditora)),
             design = design_ano1, statistic = "F"),
    error = function(e) NULL
  )

  teste_rao_ano2 <- tryCatch(
    svychisq(as.formula(paste0("~", variavel_desfecho, " + ", variavel_preditora)),
             design = design_ano2, statistic = "F"),
    error = function(e) NULL
  )

  interpretar_teste <- function(teste, ano_label) {
    if (is.null(teste)) {
      cat(sprintf("\n[!] Teste de Rao-Scott (%s) não pôde ser calculado.\n", ano_label))
      return()
    }
    F_val <- unname(teste$statistic)
    p_val <- teste$p.value
    cat(sprintf("\nTeste de Rao–Scott (%s): F = %.3f, p-valor = %.4f\n", ano_label, F_val, p_val))
    if (p_val < 0.05) {
      cat(sprintf("   ➜ Rejeitamos H₀: há evidência de associação significativa entre '%s' e '%s'.\n → as proporções de “Young” e “AMA” variam significativamente entre os grupos.\n",
                  variavel_desfecho, variavel_preditora))
    } else {
      cat(sprintf("   ➜ Não rejeitamos H₀: não há evidência estatística de associação entre '%s' e '%s'.\n",
                  variavel_desfecho, variavel_preditora))
    }
  }

  # --- imprimir resultados dos testes ---
  cat("\n------------------------------------------------------------------\n")
  cat(" Teste de Associação (Rao–Scott) considerando o desenho complexo\n")
  cat("------------------------------------------------------------------\n")
  interpretar_teste(teste_rao_ano1, ano1_label)
  interpretar_teste(teste_rao_ano2, ano2_label)

  invisible(list(
    tabela = tabela_final,
    raw = resultados,
    design = design_combinado,
    teste_rao = list(ano1 = teste_rao_ano1, ano2 = teste_rao_ano2)
  ))
}

# Explicação da Meotodologia e funções

### Metodologia Estatística

A análise foi conduzida considerando o **desenho amostral complexo** da Pesquisa Nacional de Saúde (PNS), 
com ponderações, estratos e conglomerados incorporados via o pacote `survey` em R.  
As estimativas de proporção e de **odds ratio (OR)** foram calculadas com base em **proporções ponderadas**, 
respeitando a estrutura amostral do inquérito.

#### Estimativas de Proporções

Para cada categoria da variável preditora \( X \) (por exemplo, grupo etário ou escolaridade) 
e para cada ano \( t \), estimou-se a proporção ponderada de sucesso (AMA = 1) como:

$$
\hat{p}_{X,t} = \frac{\sum_i w_i \cdot I(Y_i = 1, X_i = X, \text{ano}_i = t)}{\sum_i w_i \cdot I(X_i = X, \text{ano}_i = t)}
$$

onde \( w_i \) são os pesos amostrais ajustados e \( I(\cdot) \) é a função indicadora.

As variâncias dessas proporções foram obtidas a partir do estimador linearizado de Taylor,
usando a função `svymean`, o que garante que os intervalos de confiança refletem o 
efeito do desenho amostral.

#### Cálculo dos Odds Ratios (OR)

Para cada subgrupo, o **odds ratio** comparando AMA entre o grupo de interesse (A) 
e o grupo de referência (Y) foi calculado como:

$$
\widehat{OR} = \frac{\hat{p}_A / (1 - \hat{p}_A)}{\hat{p}_Y / (1 - \hat{p}_Y)}
$$

A variância de \( \log(\widehat{OR}) \) foi aproximada pelo método delta:

$$
\text{Var}[\log(\widehat{OR})] = 
\frac{\text{Var}[\hat{p}_A]}{(\hat{p}_A (1 - \hat{p}_A))^2} +
\frac{\text{Var}[\hat{p}_Y]}{(\hat{p}_Y (1 - \hat{p}_Y))^2} -
2 \cdot \frac{\text{Cov}(\hat{p}_A, \hat{p}_Y)}{\hat{p}_A (1 - \hat{p}_A) \hat{p}_Y (1 - \hat{p}_Y)}
$$

O **intervalo de confiança de 95%** para o OR é então:

$$
IC_{95\%} = \exp \left[ \log(\widehat{OR}) \pm 1.96 \cdot \sqrt{\text{Var}[\log(\widehat{OR})]} \right]
$$

#### Teste de Associação de Rao–Scott

Para avaliar se existe associação entre a variável de desfecho e a variável preditora
em cada ano, aplicou-se o **teste de Rao–Scott**, uma versão ajustada do teste qui-quadrado
que considera o desenho complexo.  

A hipótese nula \( H_0 \) estabelece **independência entre as variáveis**, 
e a hipótese alternativa \( H_1 \) indica **associação significativa**.

- Se \( p < 0{,}05 \): rejeita-se \( H_0 \), indicando associação significativa.  
- Se \( p \geq 0{,}05 \): não há evidência suficiente para rejeitar \( H_0 \).

#### Comparação entre anos (p-change)

Para avaliar mudanças na associação entre AMA e a variável preditora entre os anos \( t_1 \) e \( t_2 \), 
calculou-se o **p_change**, definido como o p-valor do teste da diferença entre os logaritmos dos ORs:

$$
\text{logOR}_{t} = \log \left( \frac{\hat{p}_{A,t}/(1-\hat{p}_{A,t})}{\hat{p}_{Y,t}/(1-\hat{p}_{Y,t})} \right), 
\quad 
\text{p\_change} = 2 \cdot P\left( |Z| > \frac{|\text{logOR}_{t_2} - \text{logOR}_{t_1}|}{\sqrt{\text{Var}[\text{logOR}_{t_1}] + \text{Var}[\text{logOR}_{t_2}]}} \right)
$$

Aqui, assume-se **independência entre as amostras dos anos** para simplificação.  
O p_change indica se houve uma variação estatisticamente significativa na força da associação 
(OR) entre os anos:

- Se \( p\_change < 0{,}05 \): há evidência de mudança significativa entre anos.  
- Se \( p\_change \geq 0{,}05 \): não há evidência de alteração significativa.

#### Interpretação e Evolução Temporal

Para cada subgrupo e ano, foram apresentados:

1. As proporções ponderadas de AMA (com IC 95%);
2. O **OR (AMA vs referência)**, ajustado ao desenho;
3. O **p-valor** associado ao teste de variação entre anos (**p_change**);
4. O **teste de Rao–Scott** dentro de cada ano.

Essas estatísticas permitem comparar as diferenças absolutas e relativas na prevalência
de atendimento médico adequado (AMA) entre grupos e entre anos, respeitando a
estrutura de amostragem complexa da PNS.


## Definição dos Testes

Se p < 0,05 → rejeitamos H0:
- há evidência estatística de associação entre parto_idade_avancada e Quebra de Interesse.
- → as proporções de “Young” e “AMA” variam significativamente entre os grupos.

Se p ≥ 0,05 → não rejeitamos H0:
- não há evidência estatística de associação; as diferenças observadas podem ser devidas ao acaso amostral.

In [23]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Rural", "Urbano")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "regiao_metropolitana_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Rural     |17.1           |[15.4 - 18.8]       |13.9         |[10.9 - 17.0]     |0.78 (0.60–1.02) |0.068        |14.9           |[13.8 - 16.1]       |13.8         |[11.9 - 15.7]     |0.91 (0.77–1.08) |0.275        |0.342                 |
|Urbano    |82.9           |[81.2 - 84.6]       |86.1         |[83.0 - 89.1]     |1.27 (0.98–1.66) |0.068        |85.1           

# Região Brasileira

In [24]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Sudeste", "Norte", "Nordeste", "Sul", "Centro-Oeste")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "regiao_brasileira_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria    |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Sudeste      |36.9           |[34.3 - 39.5]       |42.6         |[37.5 - 47.8]     |1.27 (1.02–1.59) |0.035*       |36.9           |[34.7 - 39.2]       |42.8         |[38.9 - 46.6]     |1.28 (1.07–1.53) |0.007*       |0.977                 |
|Norte        |10.5           |[9.4 - 11.6]        |7.0          |[5.4 - 8.7]       |0.65 (0.51–0.83) |<0.001*      |11.

# Escolaridade

In [25]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Fundamental", "Médio", "Superior", "Pós-graduação")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "escolaridade_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria     |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:-------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Fundamental   |36.1           |[33.9 - 38.2]       |35.5         |[30.8 - 40.3]     |0.98 (0.78–1.23) |0.844        |33.2           |[31.3 - 35.1]       |39.7         |[35.9 - 43.5]     |1.33 (1.11–1.58) |0.002*       |0.039*                |
|Médio         |51.0           |[48.7 - 53.2]       |32.7         |[28.0 - 37.4]     |0.47 (0.37–0.59) |<0.001*      

# Raça

In [26]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Branca", "Parda", "Preta", "Amarela", "Indígena")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "raca_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Branca    |39.5           |[37.2 - 41.7]       |49.6         |[44.5 - 54.6]     |1.51 (1.21–1.87) |<0.001*      |33.1           |[31.1 - 35.1]       |38.3         |[34.8 - 41.8]     |1.26 (1.06–1.49) |0.010*       |0.199                 |
|Parda     |49.5           |[47.4 - 51.7]       |40.7         |[35.8 - 45.6]     |0.70 (0.56–0.87) |0.001*       |52.6           

# Estado Civil


In [27]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Solteiro", "Casado", "Divorciado", "Viúvo")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "estado_civil_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria  |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:----------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Solteiro   |50.9           |[48.7 - 53.1]       |37.7         |[32.8 - 42.6]     |0.59 (0.47–0.73) |<0.001*      |50.9           |[49.0 - 52.8]       |34.7         |[31.4 - 38.0]     |0.51 (0.44–0.60) |<0.001*      |0.343                 |
|Casado     |43.1           |[41.0 - 45.3]       |53.2         |[48.3 - 58.1]     |1.50 (1.21–1.86) |<0.001*      |40.7       

# faixa_renda_per_capta

In [ ]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("<1 SM", "1 a <2 SM", "2 a <3 SM", "3 a <5 SM", "≥5 SM")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,$ git push -u origin artigo
remote: Invalid username or token. Password authentication is not supported for Git operations.
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "faixa_renda_per_capta_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|<1 SM     |72.5           |[70.5 - 74.5]       |60.4         |[55.4 - 65.3]     |0.58 (0.46–0.72) |<0.001*      |76.2           |[74.4 - 78.0]       |65.0         |[61.5 - 68.5]     |0.58 (0.49–0.69) |<0.001*      |0.976                 |
|1 a <2 SM |18.3           |[16.5 - 20.0]       |18.8         |[15.1 - 22.4]     |1.03 (0.79–1.35) |0.811        |17.0           

# IMC

In [29]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Normal", "Sobrepeso", "Obesidade", "Magreza", "Obesidade Grave")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "imc_classificacao_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria       |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Normal          |46.2           |[44.0 - 48.4]       |38.8         |[33.9 - 43.8]     |0.74 (0.59–0.92) |0.008*       |41.5           |[39.6 - 43.5]       |37.6         |[34.4 - 40.9]     |0.85 (0.72–1.00) |0.046*       |0.323                 |
|Sobrepeso       |30.5           |[28.5 - 32.5]       |35.6         |[31.1 - 40.1]     |1.26 (1.01–1.57) |0.037

# Comorbidade

In [30]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Não", "Sim")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "comorbidade_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Não       |70.9           |[69.0 - 72.8]       |56.5         |[51.6 - 61.5]     |0.53 (0.43–0.66) |<0.001*      |56.6           |[54.7 - 58.6]       |44.3         |[41.1 - 47.5]     |0.61 (0.52–0.71) |<0.001*      |0.326                 |
|Sim       |29.1           |[27.2 - 31.0]       |43.5         |[38.5 - 48.4]     |1.87 (1.51–2.32) |<0.001*      |43.4           

# Atividade Fisica

In [31]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Sim", "Não")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "atividade_fisica_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Sim       |21.9           |[20.1 - 23.8]       |21.8         |[17.4 - 26.1]     |0.99 (0.75–1.31) |0.943        |31.3           |[29.5 - 33.0]       |31.0         |[27.8 - 34.2]     |0.99 (0.84–1.16) |0.869        |0.983                 |
|Não       |78.1           |[76.2 - 79.9]       |78.2         |[73.9 - 82.6]     |1.01 (0.76–1.34) |0.943        |68.7           

# Pré Natal Adequado

In [32]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Sim", "Não")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "pre_natal_adequado_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Sim       |10.9           |[9.6 - 12.2]        |13.7         |[10.3 - 17.1]     |1.30 (0.94–1.78) |0.111        |8.4            |[7.3 - 9.4]         |9.5          |[7.6 - 11.4]      |1.15 (0.89–1.48) |0.274        |0.572                 |
|Não       |89.1           |[87.8 - 90.4]       |86.3         |[82.9 - 89.7]     |0.77 (0.56–1.06) |0.111        |91.6           

# Idade

In [45]:
library(survey)
library(dplyr)

# Média de idade e IC 95% por grupo (AMA vs Young) - 2013
idade_2013 <- svyby(
  ~idade,
  ~parto_idade_avancada,  # ajuste conforme nome real da variável AMA/Young
  design = design_maes_2013,
  FUN = svymean,
  na.rm = TRUE,
  vartype = c("se", "ci") # inclui erro padrão e IC
)

# ---- Resultados 2013 ----
idade_2013_fmt <- idade_2013 %>%
  rename(
    grupo = parto_idade_avancada,
    media_idade = idade,
    erro_padrao = se,
    ic_low = ci_l,
    ic_high = ci_u
  ) %>%
  mutate(
    grupo = ifelse(grupo == 1, "AMA (1)", "Young (0)")
  )

print("Resultados PNS 2013 - Média de Idade com IC95%")
idade_2013_fmt

# Média de idade e IC 95% por grupo (AMA vs Young) - 2013
idade_2019 <- svyby(
  ~idade,
  ~parto_idade_avancada,  # ajuste conforme nome real da variável AMA/Young
  design = design_maes_2019,
  FUN = svymean,
  na.rm = TRUE,
  vartype = c("se", "ci") # inclui erro padrão e IC
)

# ---- Resultados 2019 ----
idade_2019_fmt <- idade_2019 %>%
  rename(
    grupo = parto_idade_avancada,
    media_idade = idade,
    erro_padrao = se,
    ic_low = ci_l,
    ic_high = ci_u
  ) %>%
  mutate(
    grupo = ifelse(grupo == 1, "AMA (1)", "Young (0)")
  )

print("Resultados PNS 2019 - Média de Idade com IC95%")
idade_2019_fmt

[1] "Resultados PNS 2013 - Média de Idade com IC95%"


,grupo,media_idade,erro_padrao,ic_low,ic_high
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
0,Young (0),29.94372,0.1483882,29.65288,30.23455
1,AMA (1),41.08542,0.1732278,40.74590,41.42494


[1] "Resultados PNS 2019 - Média de Idade com IC95%"


,grupo,media_idade,erro_padrao,ic_low,ic_high
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
0,Young (0),33.16118,0.2238144,32.72251,33.59985
1,AMA (1),46.29405,0.4217347,45.46747,47.12064


In [48]:
# Combina as bases limpas
df_comb <- bind_rows(df_maes_grupo_2013_clean, df_maes_grupo_2019_clean)

# Ajusta como fator
df_comb$ano <- factor(df_comb$ano)

# Novo desenho combinado
design_comb <- svydesign(
  ids = ~UPA_PNS_num,
  strata = ~STRATO_num,
  weights = ~peso_morador_selec,
  data = df_comb,
  nest = TRUE
)

# Teste Young: 0
teste_young <- svyttest(idade ~ ano, subset(design_comb, parto_idade_avancada == 0))

# Teste AMA: 1
teste_ama <- svyttest(idade ~ ano, subset(design_comb, parto_idade_avancada == 1))


In [49]:
interpretar_teste <- function(teste, grupo_label, grupo_cod, design) {
  
  pval <- teste$p.value
  tstat <- as.numeric(teste$statistic)
  df <- as.numeric(teste$parameter)
  ic <- confint(teste)
  diff_mean <- coef(teste)

  # Médias ponderadas para cada ano
  media_2013 <- svymean(~idade,
                        subset(design, parto_idade_avancada == grupo_cod & ano == "2013"),
                        na.rm = TRUE)[1]

  media_2019 <- svymean(~idade,
                        subset(design, parto_idade_avancada == grupo_cod & ano == "2019"),
                        na.rm = TRUE)[1]

  cat("\n====================================================\n")
  cat("Teste de Diferença de Médias entre 2013 e 2019 -", grupo_label, "\n")
  cat("Método: Design-based t-test (svyttest | amostragem complexa)\n")
  cat("Hipótese nula (H0): Não houve mudança na média de idade\n")
  cat("Hipótese alternativa (H1): Houve mudança na média de idade\n\n")

  cat("Médias por ano:\n")
  cat(sprintf("  2013: %.2f anos\n", media_2013))
  cat(sprintf("  2019: %.2f anos\n\n", media_2019))

  cat(sprintf("Diferença das médias (2019 - 2013): %.2f anos\n", diff_mean))
  cat(sprintf("IC95%% da diferença: [%.2f ; %.2f]\n", ic[1], ic[2]))
  cat(sprintf("t(%d) = %.3f\n", df, tstat))
  cat(sprintf("p-valor = %.4g\n\n", pval))

  if (pval < 0.05) {
    cat("✅ Evidência estatística de mudança na idade média entre os anos.\n")
  } else {
    cat("❌ Não há evidência estatística de mudança na idade média entre os anos.\n")
  }
  cat("====================================================\n")
}

# Aplicar aos dois grupos
# interpretar_teste(teste_young, "Young (0)", grupo_cod = 0, design_comb)
interpretar_teste(teste_ama, "AMA (1)", grupo_cod = 1, design_comb)



Teste de Diferença de Médias entre 2013 e 2019 - AMA (1) 
Método: Design-based t-test (svyttest | amostragem complexa)
Hipótese nula (H0): Não houve mudança na média de idade
Hipótese alternativa (H1): Houve mudança na média de idade

Médias por ano:
  2013: 41.09 anos
  2019: 46.29 anos

IC95% da diferença: [4.32 ; 6.10]
t(2327) = 11.463
p-valor = 1.237e-29

✅ Evidência estatística de mudança na idade média entre os anos.


# peso_nascer_inferior_2_5

In [36]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Sim", "Não")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "peso_nascer_inferior_2_5_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Sim       |13.9           |[12.2 - 15.5]       |14.3         |[11.2 - 17.3]     |1.03 (0.78–1.37) |0.826        |2.0            |[1.5 - 2.4]         |3.7          |[2.1 - 5.3]       |1.91 (1.14–3.20) |0.014*       |0.041*                |
|Não       |86.1           |[84.5 - 87.8]       |85.7         |[82.7 - 88.8]     |0.97 (0.73–1.28) |0.826        |98.0           

# Prematuro

In [37]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Sim", "Não")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "prematuro_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Sim       |17.1           |[15.5 - 18.7]       |23.7         |[19.4 - 28.0]     |1.50 (1.15–1.96) |0.003*       |3.1            |[2.5 - 3.6]         |4.4          |[2.9 - 6.0]       |1.45 (0.96–2.19) |0.077        |0.885                 |
|Não       |82.9           |[81.3 - 84.5]       |76.3         |[72.0 - 80.6]     |0.66 (0.51–0.87) |0.003*       |96.9           

# Cesaria

In [38]:
# 1. Defina a ordem de referência para a variável de interesse
ordem <- c("Sim", "Não")

# 2. Chame a função
tabela <- gerar_tabela_or_evolucao(
  df_ano1 = df_maes_grupo_2013_clean,
  df_ano2 = df_maes_grupo_2019_clean,
  variavel_desfecho = "parto_idade_avancada",
  variavel_preditora = "cesaria_fct", # Use a coluna já tratada
  ordem_referencia = ordem
)


 Distribuição por coluna (colunas somam ~100%) e OR (AMA vs Young)
 Nota: AMA = 1; Young = 0. ICs e p-vals respeitam o desenho via svymean + delta



|Categoria |% Young (2013) |IC 95% (Young) 2013 |% AMA (2013) |IC 95% (AMA) 2013 |OR (95% CI) 2013 |p-valor 2013 |% Young (2019) |IC 95% (Young) 2019 |% AMA (2019) |IC 95% (AMA) 2019 |OR (95% CI) 2019 |p-valor 2019 |p_change (2013->2019) |
|:---------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:--------------|:-------------------|:------------|:-----------------|:----------------|:------------|:---------------------|
|Sim       |55.5           |[53.3 - 57.7]       |66.5         |[61.8 - 71.3]     |1.59 (1.27–2.00) |<0.001*      |12.7           |[11.4 - 14.0]       |18.3         |[15.3 - 21.2]     |1.54 (1.22–1.93) |<0.001*      |0.824                 |
|Não       |44.5           |[42.3 - 46.7]       |33.5         |[28.7 - 38.2]     |0.63 (0.50–0.79) |<0.001*      |87.3           